In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

from sklearn.tree import export_graphviz
import graphviz
import pydotplus
from IPython.display import Image

In [4]:
df = pd.read_csv('movie-data/cleaned_analysis_data.csv')

In [5]:
features = ['vote_average', 'vote_count', 'runtime', 'popularity',
            'actor_avg', 'actor_med', 'actor_dev', 'production_avg', 
            'production_med', 'production_dev', 'original_title_matches',
            'original_language_english', 'american_film', 'english_language']

In [6]:
# Drop columns that are directly related to profitability
df = df.drop(columns=['budget', 'profit', 'revenue', 'id', 'release_year', 'release_month'])

# Drop columns that are not encoded
df = df.drop(columns=["title", "release_date"])

# Define target and features
y = df['un_profitability']
X = df.drop(columns=['un_profitability'])

# Split the data into training and testing sets (70:30 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create and train the Decision Tree Classifier
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = clf.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 0.7548514851485149


In [7]:
print("Classification Report:\n", classification_report(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.81      0.81      3275
           1       0.65      0.64      0.65      1775

    accuracy                           0.75      5050
   macro avg       0.73      0.73      0.73      5050
weighted avg       0.75      0.75      0.75      5050



In [9]:
# dot_data = export_graphviz(clf, out_file=None,
#                            feature_names=features,
#                            class_names=[str(c) for c in sorted(y.unique())],
#                            filled=True, rounded=True,
#                            special_characters=True)

# graph = graphviz.Source(dot_data)
# graph.render("figures/movie_decision_tree")
# graph = pydotplus.graph_from_dot_data(dot_data)
# Image(graph.create_png())

In [ ]:
## Going to add to this the additional metrics and then then the k-hold out validation
from sklearn.model_selection import StratifiedShuffleSplit  
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)
import numpy as np
k = 10                                 # change to whatever # repeats you want
sss = StratifiedShuffleSplit(
    n_splits=k,                          # k train/test draws
    test_size=0.30,                      # 70 : 30 split each time
    random_state=42                      # reproducible shuffles
)

acc_scores, prec_scores, rec_scores, f1_scores = [], [], [], []

for train_idx, test_idx in sss.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    clf = DecisionTreeClassifier(random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc_scores.append(accuracy_score(y_test, y_pred))
    prec_scores.append(precision_score(y_test, y_pred, zero_division=0))
    rec_scores.append(recall_score(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred))

# ---------- 3.  summary  ----------
def show(name, scores):
    print(f"{name:<9}: {np.mean(scores):.3f} ± {np.std(scores):.3f}")

show("Accuracy",  acc_scores)
show("Precision", prec_scores)
show("Recall",    rec_scores)
show("F1-score",  f1_scores)

Accuracy : 0.753 ± 0.007
Precision: 0.644 ± 0.011
Recall   : 0.658 ± 0.012
F1-score : 0.651 ± 0.009
